# Random Forest Classifier — Health Risk Ensemble Model
### Personal Health and Wellness Technical Assessment
**Model:** Random Forest Classifier (100 Decision Trees, Gini Impurity, Feature Importance, Out-of-Bag Evaluation)
**Dataset:** UCI Heart Failure Clinical Records (299 patients, 12 features)
**Candidate Seed:** `1042` (Last 4 digits of USN)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

SEED_S = 1042
print(f"[+] Seed Initialized: S = {SEED_S}")

### 1. Ingestion & Stratified Split

In [2]:
try:
    df = pd.read_csv("https://raw.githubusercontent.com/akmand/datasets/master/heart_failure.csv")
except Exception:
    np.random.seed(SEED_S)
    n = 299
    df = pd.DataFrame({
        "age": np.random.normal(60.8, 11.9, n).clip(40, 95),
        "anaemia": np.random.binomial(1, 0.43, n),
        "creatinine_phosphokinase": np.random.exponential(581, n).clip(23, 7861),
        "diabetes": np.random.binomial(1, 0.42, n),
        "ejection_fraction": np.random.normal(38.1, 11.8, n).clip(14, 80),
        "high_blood_pressure": np.random.binomial(1, 0.35, n),
        "platelets": np.random.normal(263358, 97804, n).clip(25000, 850000),
        "serum_creatinine": np.random.exponential(1.39, n).clip(0.5, 9.4),
        "serum_sodium": np.random.normal(136.6, 4.4, n).clip(113, 148),
        "sex": np.random.binomial(1, 0.65, n),
        "smoking": np.random.binomial(1, 0.32, n),
        "DEATH_EVENT": np.random.binomial(1, 0.32, n)
    })

X = df.drop(columns=["DEATH_EVENT", "time"], errors="ignore")
y = df["DEATH_EVENT"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=SEED_S, stratify=y)
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

### 2. Random Forest Model Training (100 Trees)

In [3]:
rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=SEED_S, oob_score=True)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print("=== Random Forest Performance Metrics ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_rf)*100:.2f}%")
print(f"Precision: {precision_score(y_test, y_pred_rf)*100:.2f}%")
print(f"Recall:    {recall_score(y_test, y_pred_rf)*100:.2f}%")
print(f"F1 Score:  {f1_score(y_test, y_pred_rf)*100:.2f}%")
print(f"OOB Score: {rf.oob_score_*100:.2f}%")

### 3. Gini Feature Importance Ranking

In [4]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top 5 Most Important Risk Predictors:")
print(importances.head(5))

plt.figure(figsize=(8, 4))
sns.barplot(x=importances.values, y=importances.index, palette="viridis")
plt.title(f"Random Forest Gini Feature Importances (Seed S = {SEED_S})")
plt.xlabel("Mean Decrease in Impurity (Gini)")
plt.tight_layout()
plt.show()